# HEALPix Sidecar

> Generate HEALPix cell assignments for spatial data

In [ ]:
#| default_exp sidecar

In [ ]:
#| export
#| eval: false
"""healpix_sidecar.py

Create a lightweight sidecar (index-aside) that maps source geometries to HEALPix cells.

Requirements:
- dask_geopandas for lazy/parallel geoparquet reading
- cdshealpix for HEALPix computation (falls back to healpy if unavailable)
- shapely for coordinate extraction
- argparse for CLI

Output: parquet file with only two columns: source_id (original index) and healpix_id (uint64)

Usage example:
  python healpix_sidecar.py --input data.parquet --nside 64 --mode fuzzy --ncores 8
"""
from __future__ import annotations
import argparse
import logging
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    import dask_geopandas as dg
except Exception:
    raise ImportError("This script requires dask_geopandas. Install it with `pip install dask-geopandas`.")

try:
    # cdshealpix is preferred for speed
    import cdshealpix as ch
except Exception:
    ch = None

try:
    import healpy as _healpy
except Exception:
    _healpy = None

from shapely import get_coordinates  # shapely>=2.0
from shapely.geometry import Polygon, MultiPolygon
from tqdm.auto import tqdm
import antimeridian

try:
    from dask.diagnostics.progress import ProgressBar as DaskProgressBar
    DASK_PROGRESS_AVAILABLE = True
except ImportError:
    try:
        from dask.diagnostics import ProgressBar as DaskProgressBar
        DASK_PROGRESS_AVAILABLE = True
    except ImportError:
        DASK_PROGRESS_AVAILABLE = False
        DaskProgressBar = None

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("healpix_sidecar")


def compute_healpix_ids_from_lonlat(nside: int, lons: np.ndarray, lats: np.ndarray) -> np.ndarray:
    """Compute HEALPix indices for arrays of lon,lat in degrees.

    Tries to use cdshealpix if available, otherwise falls back to healpy.
    Returns a 1D integer numpy array of same length as inputs.
    """
    if lons.size == 0:
        return np.array([], dtype=np.int64)

    # normalize lons to [0,360)
    lons = np.mod(lons.astype(float), 360.0)
    lats = lats.astype(float)

    # Prefer healpy (to match notebook usage). Fall back to cdshealpix if healpy not available.
    if _healpy is not None:
        # healpy expects theta (colat) and phi (lon) in radians
        phi = np.radians(lons)
        theta = np.radians(90.0 - lats)
        return _healpy.ang2pix(nside, theta, phi, nest=True)

    if ch is not None:
        try:
            return np.asarray(ch.lonlat_to_healpix(nside, lons, lats, nest=True), dtype=np.int64)
        except Exception:
            try:
                return np.asarray(ch.lonlat_to_healpix(nside, lons, lats), dtype=np.int64)
            except Exception:
                logger.debug("cdshealpix present but call failed")

    raise RuntimeError("No HEALPix implementation available: install healpy or cdshealpix")


def process_partition(gdf, nside: int, mode: str, base_index: int | None = None, 
                     lat_min: float = -90.0, lat_max: float = 90.0,
                     lon_min: float = -180.0, lon_max: float = 180.0) -> pd.DataFrame:
    """Process a single dask partition (GeoDataFrame) and return DataFrame of assignments.

    The returned DataFrame has columns ['source_id', 'healpix_id'] and one row per assignment
    (for strict mode: at most one row per source_id; for fuzzy mode: one row per touched healpix cell).
    
    Args:
        gdf: GeoDataFrame partition
        nside: HEALPix nside parameter
        mode: 'strict' or 'fuzzy' assignment mode
        base_index: Base index for source_id generation
        lat_min: Minimum valid latitude (default: -90)
        lat_max: Maximum valid latitude (default: 90)
        lon_min: Minimum valid longitude (default: -180)
        lon_max: Maximum valid longitude (default: 180)
    """
    import pandas as _pd

    out_rows = []
    dropped_count = 0
    total_count = 0

    # if the partition is empty, return empty DataFrame
    if gdf is None or len(gdf) == 0:
        return _pd.DataFrame(columns=["source_id", "healpix_id"])

    # Determine source ids. If `base_index` is provided we generate sequential
    # global row numbers to match `gdf.reset_index()` semantics from the notebook.
    if base_index is not None:
        src_ids = base_index + np.arange(len(gdf), dtype=np.int64)
    else:
        # Prefer an explicit 'source_id' column if present; otherwise fall back to the index.
        if "source_id" in gdf.columns:
            src_ids = gdf["source_id"].to_numpy()
        else:
            src_ids = gdf.index.to_numpy()

    # iterate rows; keep work per-geometry contained to avoid large in-memory structures
    for src_id, geom in zip(src_ids, gdf.geometry.to_numpy()):
        try:
            # handle missing/empty geometries: match notebook filtering semantics by skipping
            # geometries that are None/empty rather than emitting NA rows
            if geom is None or geom.is_empty:
                continue

            # notebook pipeline: first validate the ORIGINAL geometry (pre-fix),
            # then apply antimeridian.fix_polygon and extract coordinates.
            def _is_valid_latitude(geometry):
                if geometry is None or geometry.is_empty:
                    return False
                geoms = [geometry] if getattr(geometry, "geom_type", "") == "Polygon" else list(getattr(geometry, "geoms", [geometry]))
                for g in geoms:
                    if getattr(g, "exterior", None) is not None:
                        for coord in g.exterior.coords:
                            lon = coord[0]
                            lat = coord[1]
                            if not (np.isfinite(lon) and np.isfinite(lat)):
                                return False
                            # check against configurable bounds
                            if lat < lat_min or lat > lat_max:
                                return False
                            if lon < lon_min or lon > lon_max:
                                return False
                    for interior in getattr(g, "interiors", []):
                        for coord in interior.coords:
                            lon = coord[0]
                            lat = coord[1]
                            if not (np.isfinite(lon) and np.isfinite(lat)):
                                return False
                            if lat < lat_min or lat > lat_max:
                                return False
                            if lon < lon_min or lon > lon_max:
                                return False
                return True

            total_count += 1
            # if original geometry fails the pre-fix filter, skip it entirely
            if not _is_valid_latitude(geom):
                dropped_count += 1
                continue

            # apply the antimeridian fix (not used for the pre-filter decision)
            try:
                geom2 = antimeridian.fix_polygon(geom)
            except Exception:
                geom2 = geom

            # vectored extraction using shapely.get_coordinates when available (on the fixed geometry)
            try:
                coords = get_coordinates(geom2)
            except Exception:
                # fallback gather exterior + interiors for polygons/multipolygons
                coords_list = []
                if isinstance(geom2, Polygon):
                    coords_list.extend(np.asarray(geom2.exterior.coords, dtype=float))
                    for r in geom2.interiors:
                        coords_list.extend(np.asarray(r.coords, dtype=float))
                elif isinstance(geom2, MultiPolygon):
                    for part in geom2.geoms:
                        coords_list.extend(np.asarray(part.exterior.coords, dtype=float))
                        for r in part.interiors:
                            coords_list.extend(np.asarray(r.coords, dtype=float))
                else:
                    # generic fallback: treat as single-vertex geometry
                    try:
                        coords_list = np.asarray(list(geom2.coords), dtype=float)
                    except Exception:
                        coords_list = []

                if len(coords_list) == 0:
                    # no coordinates to assign -> skip (matches notebook behaviour where empty lists
                    # later produce no exploded rows)
                    continue
                coords = np.asarray(coords_list, dtype=float)

            if coords.size == 0:
                continue

            # normalize longitudes to [0,360) for HEALPix computation
            lons = coords[:, 0].astype(float)
            lats = coords[:, 1].astype(float)

            # filter invalid lat/lon values using configurable bounds
            mask = (np.isfinite(lons) & np.isfinite(lats) & 
                   (lats >= lat_min) & (lats <= lat_max) &
                   (lons >= lon_min) & (lons <= lon_max))
            if not np.any(mask):
                dropped_count += 1
                continue

            lons = lons[mask]
            lats = lats[mask]

            # compute healpix indices for all vertices
            hids = compute_healpix_ids_from_lonlat(nside, lons, lats)
            if hids.size == 0:
                continue

            unique = np.unique(hids)
            if mode == "strict":
                # only accept if all vertices fall in same single HEALPix cell
                if unique.size == 1:
                    out_rows.append({"source_id": int(src_id), "healpix_id": int(unique[0])})
            else:
                # fuzzy: replicate source for each unique cell
                for hid in unique:
                    out_rows.append({"source_id": int(src_id), "healpix_id": int(hid)})

        except Exception as e:
            # protect partition processing from crashing; log and continue
            logger.debug(f"skipping source {src_id} due to error: {e}")
            out_rows.append({"source_id": int(src_id) if src_id is not None else pd.NA, "healpix_id": pd.NA})
            total_count += 1
            dropped_count += 1
            continue

    # Log statistics for this partition
    if total_count > 0:
        drop_pct = 100.0 * dropped_count / total_count
        if dropped_count > 0:
            logger.info(f"Partition: processed {total_count} geometries, dropped {dropped_count} ({drop_pct:.1f}%) due to invalid bounds")
    
    if len(out_rows) == 0:
        return _pd.DataFrame(columns=["source_id", "healpix_id"])  # empty
    df_out = _pd.DataFrame(out_rows)
    # ensure types
    # source_id should be signed int64 to match geopandas.reset_index() behavior
    df_out["source_id"] = df_out["source_id"].astype(np.int64)
    # healpix ids are non-negative; use pandas nullable unsigned integer so missing values are allowed
    df_out["healpix_id"] = df_out["healpix_id"].astype("UInt64")
    return df_out


def build_output_path(input_path: Path, mode: str, nside: int) -> Path:
    stem = input_path.stem
    # separate the key:values with - and _ between them 
    outname = f"{stem}.cell-healpix_assignment-{mode}_nside-{nside}_order-nested.parquet"
    return input_path.with_name(outname)


def main(argv=None):
    parser = argparse.ArgumentParser(description="Create HEALPix sidecar mapping source geometries to cells.")
    parser.add_argument("--input", "-i", required=True, help="Path to input GeoParquet file")
    parser.add_argument("--nside", "-n", type=int, nargs='+', required=True, help="One or more HEALPix nside values (powers of 2). Example: -n 64 128")
    parser.add_argument("--mode", "-m", choices=["strict", "fuzzy"], default="fuzzy",
                        help="Assignment mode: strict (single-cell only) or fuzzy (all cells touched)")
    parser.add_argument("--ncores", type=int, default=max(1, (os.cpu_count() or 2) - 1),
                        help="Number of cores to use for Dask workers (defaults to cpu_count-1)")
    parser.add_argument("--output_dir", "-o", default=None,
                        help="Directory to write the output file (defaults to same folder as input)")
    parser.add_argument("--no-coalesce", dest="no_coalesce", action="store_true",
                        help="Do not coalesce partitions into a single file; write partitioned parquet (default: coalesce to single file)")
    parser.add_argument("--lat-min", type=float, default=-90.0,
                        help="Minimum valid latitude for geometry filtering (default: -90)")
    parser.add_argument("--lat-max", type=float, default=90.0,
                        help="Maximum valid latitude for geometry filtering (default: 90)")
    parser.add_argument("--lon-min", type=float, default=-180.0,
                        help="Minimum valid longitude for geometry filtering (default: -180)")
    parser.add_argument("--lon-max", type=float, default=180.0,
                        help="Maximum valid longitude for geometry filtering (default: 180)")
    parser.add_argument("--loglevel", "-l", choices=["debug", "info", "warning", "error"], default="info",
                        help="Set logging level (default: info)")
    args = parser.parse_args(argv)

    input_path = Path(args.input)
    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        sys.exit(2)

    # validate nsides
    nsides = args.nside
    for n in nsides:
        if n <= 0 or (n & (n - 1)) != 0:
            logger.error("nside must be a positive power of two: invalid value %s", n)
            sys.exit(2)

    logger.info(f"Reading input lazily from {input_path}; ncores={args.ncores}; mode={args.mode}; nsides={nsides}")
    logger.info(f"Geometry bounds: lat=[{args.lat_min}, {args.lat_max}], lon=[{args.lon_min}, {args.lon_max}]")

    # configure logging level from CLI
    level_map = {
        'debug': logging.DEBUG,
        'info': logging.INFO,
        'warning': logging.WARNING,
        'error': logging.ERROR,
    }
    root_level = level_map.get(args.loglevel, logging.INFO)
    logging.getLogger().setLevel(root_level)
    logger.setLevel(root_level)

    # read lazily with dask_geopandas
    # let dask decide partitions but hint with npartitions based on ncores
    try:
        ddf = dg.read_parquet(str(input_path))
    except Exception:
        # try forcing to use dask read with explicit npartitions
        ddf = dg.read_parquet(str(input_path), npartitions=args.ncores)

    # We will compute explicit, global `source_id` values (0..N-1) that match
    # `gdf.reset_index()` by computing per-partition offsets and passing them to
    # `process_partition` as `base_index`.

    # apply partition-level processing
    # map_partitions will return a dask DataFrame; ensure meta is correct
    # Dask/meta: use pandas-recognized dtype strings to avoid version-specific dtype issues
    meta = pd.DataFrame({"source_id": pd.Series(dtype="int64"), "healpix_id": pd.Series(dtype="UInt64")})

    logger.info("Starting partitioned HEALPix assignment (this may take time)")

    # Loop over requested nsides without reloading the input (show overall progress)
    for nside in tqdm(nsides, desc="nsides", unit="nside"):
        logger.info("Processing nside=%s", nside)
        # We'll use delayed partitions so we can compute global source_id offsets
        import dask
        delayed_partitions = ddf.to_delayed()
        # compute partition lengths to derive base offsets for source_id
        try:
            part_lengths = ddf.map_partitions(lambda df: len(df)).compute().tolist()
        except Exception:
            # fallback: compute lengths by materializing delayed partitions (may be slower)
            part_lengths = [int(dask.compute(dask.delayed(lambda df: len(df))(p))[0]) for p in delayed_partitions]

        if len(part_lengths) != len(delayed_partitions):
            # defensive: if mismatch, fallback to equal-sized offsets (best-effort)
            logger.warning("Partition count mismatch; falling back to equal offsets")
            part_lengths = [len(delayed_partitions)] * len(delayed_partitions)

        offsets = np.concatenate(([0], np.cumsum(part_lengths)[:-1]))

        # build delayed tasks that pass the base_index per partition so source_id
        # corresponds to global row numbering (like geopandas.reset_index())
        tasks = [dask.delayed(process_partition)(part, nside, args.mode, int(offsets[i]),
                                                  args.lat_min, args.lat_max, args.lon_min, args.lon_max)
                 for i, part in enumerate(delayed_partitions)]
        nparts = len(tasks)

        # If user requested partitioned (no coalesce) output, compute each delayed
        # partition and write a separate parquet file per partition while preserving
        # global `source_id` numbering.
        if args.no_coalesce:
            out_part_dir = out_file.with_suffix('.parts')
            out_part_dir.mkdir(parents=True, exist_ok=True)
            logger.info("Writing partitioned parquet to %s (no coalesce requested)", out_part_dir)
            parts = dask.compute(*tasks)
            for idx, df_part in enumerate(parts):
                if df_part is None or len(df_part) == 0:
                    continue
                df_part = df_part.astype({"source_id": "int64", "healpix_id": "UInt64"})
                part_path = out_part_dir / f"part-{idx:06d}.parquet"
                try:
                    df_part.to_parquet(str(part_path), engine="pyarrow", index=False)
                except Exception:
                    # fallback: write via pyarrow Table to ensure metadata compatibility
                    try:
                        import pyarrow as pa
                        table = pa.Table.from_pandas(df_part, preserve_index=False)
                        import pyarrow.parquet as pq
                        pq.write_table(table, str(part_path))
                    except Exception:
                        logger.exception("Failed to write partitioned parquet for %s", part_path)
            logger.info("Wrote partitioned parquet to %s", out_part_dir)
            continue

        # prepare output path (single file target by default)
        if args.output_dir:
            out_dir = Path(args.output_dir)
            out_dir.mkdir(parents=True, exist_ok=True)
        else:
            out_dir = input_path.parent

        out_file = out_dir / build_output_path(input_path, args.mode, nside).name

        # NOTE: we will handle no-coalesce using the delayed `tasks` below so that
        # `source_id` reflects global row numbers. The old `result.to_parquet`
        # approach could write inconsistent `source_id` values when using Dask.

        logger.info(f"Computing partitions and writing single parquet file to {out_file}")

        # compute partitions in batches and write incrementally using pyarrow ParquetWriter
        try:
            import dask
            import pyarrow as pa
            import pyarrow.parquet as pq
        except Exception as e:
            logger.error("pyarrow and dask are required for coalescing partitions to a single file: %s", e)
            logger.info("Falling back to writing partitioned parquet folder instead")
            out_part_dir = str(out_file.with_suffix('.parts'))
            result.to_parquet(out_part_dir, engine="pyarrow", write_index=False)
            logger.info("Wrote partitioned parquet to %s", out_part_dir)
            continue

        # tasks is the list of delayed partition computations

        if nparts == 0:
            # nothing to write; write empty file with explicit schema and metadata
            schema = pa.schema([
                ("source_id", pa.int64()),
                ("healpix_id", pa.uint64()),
            ])
            # add file-level metadata (use a distinct name to avoid clobbering Dask meta)
            pq_meta = {"nside": str(nside), "mode": args.mode, "order": "nested"}
            schema = schema.with_metadata({k: v.encode() for k, v in pq_meta.items()})
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=["source_id", "healpix_id"]).astype({"source_id": "int64", "healpix_id": "UInt64"}), schema=schema, preserve_index=False)
            pq.write_table(empty_table, str(out_file))
            logger.info("Wrote empty output %s", out_file)
            continue

        # create an explicit pyarrow schema that we'll use for the ParquetWriter and embed metadata
        schema = pa.schema([
            ("source_id", pa.int64()),
            ("healpix_id", pa.uint64()),
        ])
        pq_meta = {"nside": str(nside), "mode": args.mode, "order": "nested"}
        schema = schema.with_metadata({k: v.encode() for k, v in pq_meta.items()})


        batch_size = max(1, args.ncores)
        writer = None
        total_rows_written = 0

        # Per-nside progress bar that advances as Dask computes partition batches.
        pbar = tqdm(total=nparts, desc=f"nside={nside}", unit="part", position=0, leave=True)
        try:
            for i in range(0, nparts, batch_size):
                batch = tasks[i : i + batch_size]
                batch_start = i + 1
                batch_end = min(i + batch_size, nparts)
                pbar.set_description(f"nside={nside} [{batch_start}-{batch_end}/{nparts}]")
                
                # compute this batch with progress tracking if available
                if DASK_PROGRESS_AVAILABLE and logger.level <= logging.INFO:
                    with DaskProgressBar():
                        res = dask.compute(*batch)
                else:
                    res = dask.compute(*batch)
                # res is a tuple of pandas DataFrames
                non_empty = [r for r in res if (r is not None and len(r) > 0)]
                if not non_empty:
                    pbar.update(len(batch))
                    continue
                df_batch = pd.concat(non_empty, ignore_index=True)
                # ensure dtypes (use pandas nullable UInt64 for healpix_id to allow missing)
                df_batch = df_batch.astype({"source_id": "int64", "healpix_id": "UInt64"})
                # create arrow table with enforced schema/metadata
                table = pa.Table.from_pandas(df_batch, schema=schema, preserve_index=False)
                batch_rows = len(df_batch)
                total_rows_written += batch_rows
                
                if writer is None:
                    # overwrite existing single file if present
                    if out_file.exists():
                        try:
                            out_file.unlink()
                        except Exception:
                            logger.warning("Could not remove existing output file %s", out_file)
                    writer = pq.ParquetWriter(str(out_file), schema)
                writer.write_table(table)
                
                # update progress with statistics
                pbar.set_postfix({
                    'rows': total_rows_written,
                    'batch_rows': batch_rows
                })
                pbar.update(len(batch))
        finally:
            pbar.close()

        # close the writer if we created one
        if writer is not None:
            writer.close()
            logger.info("Wrote single parquet file: %s", out_file)
        else:
            # if nothing was written, write empty file with schema
            empty_table = pa.Table.from_pandas(pd.DataFrame(columns=["source_id", "healpix_id"]).astype({"source_id": "int64", "healpix_id": "UInt64"}), schema=schema, preserve_index=False)
            pq.write_table(empty_table, str(out_file))
            logger.info("Wrote empty output %s", out_file)

    # end loop over nsides



# CLI entry point (use via command line or import main() function)


## Usage Example

See the `main()` function for CLI usage, or import functions directly for programmatic use.